# Data Analysis and Visualisation

## Lab 4: Gaps and merge

### Preliminaries

In [ ]:
import pandas as pd

### Part 1

#### Task 1: Load the loans and look for gaps

In [ ]:
loans = pd.read_csv("data/library_loans_clean.csv", parse_dates=["loan_date", "return_date"])

print(loans.shape)
print(loans.dtypes)
print(loans.isna().sum())
print((loans.isna().mean() * 100).round(1))

(600, 8)
loan_id                   str
member_id                 str
title                     str
genre                     str
loan_date      datetime64[us]
return_date    datetime64[us]
late_fee              float64
returned                 bool
dtype: object
loan_id         0
member_id       0
title           0
genre           0
loan_date       3
return_date    36
late_fee        0
returned        0
dtype: int64
loan_id        0.0
member_id      0.0
title          0.0
genre          0.0
loan_date      0.5
return_date    6.0
late_fee       0.0
returned       0.0
dtype: float64


Only two columns have gaps: loan_date (3 missing, 0.5 %) and return_date (36 missing, 6.0 %).

#### Task 2: Two kinds of missing

In [ ]:
display(loans[loans["loan_date"].isna()])

print((loans["return_date"].isna() & loans["returned"]).sum())
print((loans["return_date"].isna() & ~loans["returned"]).sum())

,loan_id,member_id,title,genre,loan_date,return_date,late_fee,returned
462,L-30463,M0007,Blue Moon Protocol,Sci-fi,NaT,2026-05-24,0.0,True
467,L-30468,M0018,Blue Moon Protocol,Sci-fi,NaT,2026-06-08,0.0,True
571,L-30572,M0090,Mimmi and the Moon,Children,NaT,2026-06-26,0.0,True


6
30


30 loans have no return date and the book is not back: missing for a reason, nobody has returned it yet. 6 have no return date but the book is back: missing by accident, the return date should exist and was not recorded.

In [ ]:
loans["has_return_date"] = loans["return_date"].notna()
print(loans["has_return_date"].value_counts())

has_return_date
True     564
False     36
Name: count, dtype: int64


#### Task 3: Look at the members

In [ ]:
members_raw = pd.read_csv("data/library_members.csv")
members = members_raw.copy()

print(members.shape)
members.info()
print(members["birth_year"].describe())

(148, 3)
<class 'pandas.DataFrame'>
RangeIndex: 148 entries, 0 to 147
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   member_id   148 non-null    str    
 1   branch      136 non-null    str    
 2   birth_year  130 non-null    float64
dtypes: float64(1), str(2)
memory usage: 3.6 KB
count     130.000000
mean     1983.384615
std        25.158223
min      1900.000000
25%      1978.250000
50%      1988.000000
75%      1996.750000
max      2030.000000
Name: birth_year, dtype: float64


In [ ]:
print("Wrong birth year:", "1900:", (members['birth_year'] == 1900).sum(), ", 2030:", (members['birth_year'] == 2030).sum())
print("Duplicates:", members.duplicated().sum(), members.duplicated(subset='member_id').sum())

Wrong birth year: 1900: 8 , 2030: 3
Duplicates: 4 4


Problems: `branch` has 12 missing, `birth_year` has 18 missing, the minimum 1900 (8 members) looks like a placeholder, the maximum 2030 (3 members) is in the future, and 4 members are in the table twice (4 duplicate rows and 4 duplicate `member_ids`).

#### Task 4: Placeholders and impossible values

In [ ]:
members['birth_year'] = members['birth_year'].where(members['birth_year'].between(1920, 2015))

print(members['birth_year'].isna().sum())
print(members['birth_year'].describe())

29
count     119.000000
mean     1987.815126
std        12.062666
min      1957.000000
25%      1979.000000
50%      1988.000000
75%      1996.500000
max      2010.000000
Name: birth_year, dtype: float64


1900 and 2030 are not real birth years, so they became NaN. Now 29 are missing (18 + 8 + 3). The minimum is 1957 and the maximum 2010, and the mean rose from 1983.38 to 1987.82.